# 06 — Entrepreneurship Variables

Construct business entry and survival variables.

In [ ]:
import sys
sys.path.insert(0, '..')

from src.data_loader import load_data
from src.cleaning import standardise_columns
from src.entrepreneurship import add_business_entry, compute_business_survival

df = standardise_columns(load_data('synthetic'))
df = add_business_entry(df)
survival = compute_business_survival(df)
survival.head(10)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4))
sns.histplot(survival['survival_years'], bins=range(1, 10), ax=ax)
ax.set_title('Distribution of (synthetic) business survival years')
plt.show()

## Survival analysis

The histogram above is descriptive only. For a proper survival model --
accounting for right-censoring (businesses still active at the end of the
study window) -- fit a Kaplan-Meier curve and a Cox proportional-hazards
model against founder covariates at entry.

In [ ]:
from src.entrepreneurship import compute_business_survival_with_covariates
from src.survival import add_event_flag, fit_kaplan_meier, fit_kaplan_meier_by_group, fit_cox_ph

survival_cov = compute_business_survival_with_covariates(
    df, covariate_cols=['age', 'sex', 'chronic_condition'],
)

study_end_year = df['year'].max()
survival_cov = add_event_flag(survival_cov, study_end_year=study_end_year)
survival_cov[['business_id', 'entry_year', 'last_observed_year', 'survival_years', 'event_observed']].head()


In [ ]:
kmf = fit_kaplan_meier(survival_cov)
ax = kmf.plot_survival_function()
ax.set_title('Kaplan-Meier: overall business survival (synthetic)')


In [ ]:
km_by_chronic = fit_kaplan_meier_by_group(survival_cov, group_col='chronic_condition')
fig, ax = plt.subplots(figsize=(7, 4))
for group_value, fitter in km_by_chronic.items():
    fitter.plot_survival_function(ax=ax)
ax.set_title('Business survival by founder chronic condition status at entry (synthetic)')
plt.show()


In [ ]:
cph = fit_cox_ph(survival_cov, covariates=['chronic_condition', 'age'])
cph.print_summary()
